# えんじいろ 文章変換（Colab）

Google AI Studio の Gemini 3.5 Flash Lite を使って、文章を
赤ちゃん・園児語 または ママ・やさしい口調 へ言い換えます。
あわせて、投稿してよいかの判定（モデレーション）も行います。

**このノートブックは `ai/moderation_rules.py` と `ai/transform_api.py` から
生成しています。** 仕様を直すときは `.py` 側を直してください。

**必要なもの**
- Google AI Studio で取得した APIキー
- インターネット接続

**使い方**
1. 「ランタイム」→「セッションを再起動」
2. 下のセルを実行し、APIキーを入力する

**モデルについて**
Issue #15 の当初指定は Gemma 4 31B でしたが、知能が不足していたため、
人間監督の判断で Gemini 3.5 Flash Lite へ変更しました。


In [ ]:
# ============================================================
# えんじいろ 文章変換（Colab用・1セル完結）
#
# ai/moderation_rules.py と ai/transform_api.py から生成しています。
# 直すときは .py 側を直してください。
#
# 先に「ランタイム」→「セッションを再起動」してから実行してください。
# ============================================================
!pip install -q -U google-genai janome

import getpass
import json
import os
import re
import sys
import time
import unicodedata
import urllib.error
import urllib.request
from typing import Any, Literal

from google import genai
from google.genai import types

COLAB_CLIENT = genai.Client(api_key=getpass.getpass("Google AI Studio APIキー: "))

import re
import sys
import unicodedata


# ============================================================
# 1. NG辞書
# ============================================================
# 人間が運用で育てる前提の初期値。ここに挙げたものが完全な一覧ではない。
# 追加・削除は人間監督の判断で行うこと。AIが勝手に増やさない。

# 値は「その語をどう照合するか」。
#   None    … 正規化した文字列への部分一致。形態素解析がなくても効く
#   ANY_POS … 形態素解析して、1語として完全一致したときだけ拾う
#   "名詞"  … 上に加えて、その品詞のときだけ拾う
#
# 部分一致だと日常語に当たってしまう語に、形態素解析を使う。
#   「しね」→「推しねこ」／「ころす」→「石ころすら」
#   「バカ」→「ばかり」（助詞）／「クズ」→「くずれる」（動詞）

ANY_POS = "*"

# 辞書ファイルに書く照合方法の名前と、内部表現の対応。
_RULE_NAMES = {"substring": None, "token": ANY_POS}

# Colab など、ファイルを読めない場所で使うときの埋め込み先。
# 生成スクリプトがここへ辞書を流し込む。空なら dictionaries/ から読む。
INLINE_DICTIONARIES = {
    'block': {
        '死ね': None,
        '殺す': None,
        '消えろ': None,
        'キチガイ': None,
        'ガイジ': None,
        'きえろ': None,
        'きちがい': None,
        'しね': '*',
        'ころす': '*',
        'がいじ': '*',
    },
    'rewrite': {
        '無能': None,
        '役立たず': None,
        'バカ': '名詞',
        '馬鹿': '名詞',
        'カス': '名詞',
        'クズ': '名詞',
        'ボケ': '名詞',
        'マヌケ': '名詞',
    },
    'self_harm': {
        '死にたい': '*',
        'しにたい': '*',
        '死のう': '*',
        'しのう': '*',
        '消えたい': '*',
        'きえたい': '*',
        '生きていたくない': '*',
        '生きるのをやめる': '*',
        '自殺': '*',
        '自傷': '*',
        '自害': '*',
        'リストカット': '*',
        'リスカ': '*',
        '首を吊る': '*',
        '首吊り': '*',
        '飛び降りる': '*',
        '練炭': '*',
        'オーバードーズ': '*',
    },
    'harm_others': {
        '殺してやる': '*',
        '殺害': '*',
        '殺人': '*',
        '放火': '*',
        '爆破': '*',
        '通り魔': '*',
        '刺してやる': '*',
        '殴ってやる': '*',
        'ぶっ殺': '*',
    },
}


def load_dictionary(name: str) -> dict:
    """dictionaries/<name>.txt を読み込む。

    1行1語。空行と # で始まる行は無視。
    「語<TAB>照合方法」の形式で、照合方法を省くと substring になる。
    書き方の詳細は dictionaries/README.md を参照。
    """
    if name in INLINE_DICTIONARIES:
        return dict(INLINE_DICTIONARIES[name])

    from pathlib import Path

    # Colab のセルなど、ファイルとして実行されていない場所では __file__ が無い。
    # その場合は埋め込みしか使えないので、ここで諦める。
    module_file = globals().get("__file__")
    if not module_file:
        print(
            f"[警告] 辞書 {name} を読み込めません。"
            "埋め込みもファイルもありません。",
            file=sys.stderr,
        )
        return {}

    path = Path(module_file).resolve().parent / "dictionaries" / f"{name}.txt"
    if not path.exists():
        print(f"[警告] 辞書が見つかりません: {path}", file=sys.stderr)
        return {}

    table = {}
    for lineno, raw in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        word, _, rule_name = line.partition("\t")
        word = word.strip()
        rule_name = rule_name.strip() or "substring"
        if not word:
            continue
        # substring / token は決まった値へ、それ以外は品詞名としてそのまま使う
        table[word] = _RULE_NAMES.get(rule_name, rule_name)
    return table


# どちらの辞書に入れるかは、人間監督の決めた基準に従う。
#
#   言い換えて愚痴や励ましになるなら  → NG_WORDS_REWRITE
#   どう言い換えても前向きにならないなら → NG_WORDS_BLOCK
#
# 「無能」は「うまくいかなくて困っている」に言い換えられるが、
# 「死ね」はやわらげようとすると中身が何も残らない。

# どう言い換えても前向きな文章にならないもの
NG_WORDS_BLOCK = load_dictionary("block")

# 言い換えれば愚痴や励ましになるもの（マサカリ寄りの語）
NG_WORDS_REWRITE = load_dictionary("rewrite")

# 「ゴミ」と「アホ」はここに入れない。
#   ゴミ … 罵倒も「ごみを捨てる」も名詞。直後の語でも分けられない
#   アホ … 「アホらしい」が アホ[名詞]＋らしく[助動詞] に分かれ、
#          「あいつはバカだ」の バカ[名詞]＋だ[助動詞] と同じ形になる
# いずれも実測で確認済み。文脈を見ないと判定できないので、
# LLM側のマサカリ判定に任せる。

# 自傷を示す表現。
#
# 人間監督の決定（TBD-9 の回答）:
#   「自傷・他害は絶対に弾いてください。ここは犯罪者・自殺者応援サイトでは
#     ないのです」
#
# したがって、上の「言い換えて愚痴になるなら rewrite_required」という基準を
# ここには適用しない。言い換えられそうに見えても block とする。
# この扱いを変えるのは人間監督だけ。AIの判断で緩めない。
#
# 一方で「つらい」「しんどい」「もう限界」といった弱音は、
# えんじいろが書くための場所として用意しているものなので、ここには入れない。
# 「死ぬ」は入れない。「サーバーが死んだ」「プロセスが死んでる」という
# 言い方をエンジニアは日常的に使う。原形で照合すると全部巻き込む。
SELF_HARM_WORDS = load_dictionary("self_harm")

# 他人を傷つけること、犯罪をほのめかす表現。
# 「殺す」「ころす」は NG_WORDS_BLOCK にも入っているが、
# 理由コードを分けたいのでここにも置く。
#
# 「刺す」「殴る」は入れない。「釘を刺す」「壁を殴る」と区別できないため。
# この種のものは LLM 側の判定に任せる。
HARM_OTHERS_WORDS = load_dictionary("harm_others")

# 自傷・他害を検出したときの扱い。人間監督の決定により block で固定。
# 設定値として残してあるが、AIの判断で変更しないこと。
SELF_HARM_ACTION = "block"
HARM_OTHERS_ACTION = "block"


# ============================================================
# 2. 伏字回避の正規化
# ============================================================

# 幅ゼロ文字・制御文字
_INVISIBLE = re.compile(r"[​-‏‪-‮⁠-⁤﻿­]")

# 伏字に使われやすい記号。文字と文字の間に挟んで検出を逃れる用途を想定する。
_MASK_CHARS = "○●◯〇◎＊*✳✱×✕╳・･.,、。_＿-－ー‐―~〜^ 　\t"
_MASK_PATTERN = re.compile("[" + re.escape(_MASK_CHARS) + "]+")

# 形態素解析へ渡す前の下ごしらえ用。句読点は残す。
# 「、」「。」まで消すと文が繋がって、解析器が語の切れ目を見誤るため。
_MASK_CHARS_LIGHT = "○●◯〇◎＊*✳✱×✕╳・･_＿^ 　\t"
_MASK_PATTERN_LIGHT = re.compile("[" + re.escape(_MASK_CHARS_LIGHT) + "]+")


def normalize_for_check(text: str) -> str:
    """判定用の文字列を作る。表示には使わない。

    - 全角英数字・半角カナなどを NFKC でそろえる（ﾀﾞﾒ → ダメ）
    - 幅ゼロ文字を落とす
    - 伏字記号を落として「し ね」「し○ね」を「しね」にする
    - 3文字以上の繰り返しを2文字にたたむ
    - カタカナをひらがなにそろえる

    濁点そのものは落とさない。落とすと「ダメ」が「ため」になり、
    「バカ」が「はか」になって「はかる」に一致するなど、
    誤検出のほうが害が大きいため。
    「し゛ね」のような濁点を使った回避は、この関数では防げない。
    """
    if not text:
        return ""

    normalized = unicodedata.normalize("NFKC", text)
    normalized = _INVISIBLE.sub("", normalized)
    normalized = normalized.lower()
    normalized = _MASK_PATTERN.sub("", normalized)

    # 「あああああ」→「filtered」のような引き延ばしをたたむ
    normalized = re.sub(r"(.)\1{2,}", r"\1\1", normalized)

    # カタカナ → ひらがな
    normalized = "".join(
        chr(ord(ch) - 0x60) if "ァ" <= ch <= "ヶ" else ch
        for ch in normalized
    )
    return normalized


def normalize_for_tokenize(text: str) -> str:
    """形態素解析へ渡す前の下ごしらえ。

    伏字記号は落とすが、句読点と文字種はそのまま残す。
    「バ○カ」を「バカ」に戻して解析器に渡すのが目的。
    aggressive な normalize_for_check を通すと句読点まで消えて、
    解析器が語の切れ目を見誤る。
    """
    if not text:
        return ""
    normalized = unicodedata.normalize("NFKC", text)
    normalized = _INVISIBLE.sub("", normalized)
    normalized = _MASK_PATTERN_LIGHT.sub("", normalized)
    # 「しねええええ」を「しね」に戻す。ここでたたまないと
    # 解析器が し|ねえ|え|ええ に割ってしまい、1語として照合できない。
    # 日本語で同じ文字が3つ以上続くことは少ないので、1文字まで落とす。
    return re.sub(r"(.)\1{2,}", r"\1", normalized)


# ============================================================
# 3. 形態素解析（任意）
# ============================================================
# janome が入っていれば品詞を見た判定を行う。入っていなければ、
# 品詞指定のない語だけを部分一致で拾う。
#
#     python -m pip install janome
#
# 依存を必須にしないのは、Issue #15 が ai/requirements.txt の変更を
# 禁じているため。入れれば精度が上がる、という位置づけにしてある。

_tokenizer = None
_tokenizer_tried = False


def tokenizer_available() -> bool:
    """形態素解析器が使えるか。初回だけ読み込みを試す。

    使えない場合は一度だけ警告を出す。黙って精度が落ちると、
    見逃しているのに動いているように見えてしまうため。
    """
    global _tokenizer, _tokenizer_tried
    if not _tokenizer_tried:
        _tokenizer_tried = True
        try:
            from janome.tokenizer import Tokenizer
            _tokenizer = Tokenizer()
        except ImportError:
            _tokenizer = None
            print(
                "[警告] janome が入っていないため、品詞を見た判定を行いません。\n"
                "        ひらがな表記のNG語（しね など）を見逃します。\n"
                "        python -m pip install janome",
                file=sys.stderr,
            )
    return _tokenizer is not None


def iter_tokens(text: str) -> list[tuple[str, str, str]]:
    """(表層形, 品詞, 原形) の一覧を返す。解析器がなければ空。

    原形も返すのは、活用で表層形が変わる語を拾うため。
    「首を吊ろうとした」の「吊ろ」は原形が「吊る」になる。
    """
    if not tokenizer_available():
        return []
    return [
        (token.surface, token.part_of_speech.split(",")[0], token.base_form)
        for token in _tokenizer.tokenize(normalize_for_tokenize(text))
    ]


# 一致した語を打ち消す条件。いずれも直後のトークンを見る。
#
# 罵倒として使うとき、その語のうしろには助詞や助動詞が来る。
#   あいつはバカだ ／ このボケが ／ あんなのクズだ
# 一方、名詞や形容詞が続くときは複合語の一部である。
#   バカでかい ／ ボケ防止 ／ クズ野菜 ／ 馬鹿丁寧
_COMPOUND_POS = ("名詞", "形容詞")

# 「て」「で」「ば」が続くときは、名詞ではなく動詞の活用形。
#   写真がボケていた ／ しねばよかった
_VERB_ENDINGS = ("て", "で", "ば")


def _is_conjugation(tokens: list, index: int) -> bool:
    """直後が て・で・ば なら、名詞ではなく動詞の活用形とみなす。

    どの語にも当てはめてよい。
    「写真がボケていた」「しねばよかった」を除外するためのもの。
    """
    if index + 1 >= len(tokens):
        return False
    return tokens[index + 1][0] in _VERB_ENDINGS


def _is_compound(tokens: list, index: int) -> bool:
    """直後が名詞・形容詞なら、複合語の一部とみなす。

    これは罵倒に使う名詞（バカ・クズ・ボケ）の曖昧さを解くための規則で、
    品詞を指定した語にだけ当てはめる。
    自傷・他害の語に当てはめてはいけない。
    「殺害予告」「しにたい気分」まで無害と判定してしまう（実測で確認）。
    """
    if index + 1 >= len(tokens):
        return False
    return tokens[index + 1][1].startswith(_COMPOUND_POS)


# 「消えたい」のように、解析すると複数のトークンに分かれる語がある。
#   消えたい      → 消え[動詞] | たい[助動詞]
#   消えたいくつか → 消え[動詞] | た[助動詞] | いくつか[名詞]
# 続きをつないで照合すれば、この2つを取り違えずに済む。
MAX_TOKEN_WINDOW = 5


def _match_by_tokens(text: str, tokenized: dict) -> list[str]:
    """連続するトークンをつないで、辞書の語と一致するかを調べる。"""
    hits = []
    wanted = {normalize_for_check(w): (w, r) for w, r in tokenized.items()}
    tokens = iter_tokens(text)

    for start in range(len(tokens)):
        joined = ""
        for end in range(start, min(start + MAX_TOKEN_WINDOW, len(tokens))):
            prefix = joined
            joined += normalize_for_check(tokens[end][0])
            # 表層形でも原形でも照合する。最後の語だけ活用が変わるため、
            # 原形に差し替えるのは末尾のトークンだけでよい。
            with_base = prefix + normalize_for_check(tokens[end][2])
            found = wanted.get(joined) or wanted.get(with_base)
            if not found:
                continue
            word, rule = found
            if _is_conjugation(tokens, end):
                continue
            if rule != ANY_POS:
                # 品詞は先頭のトークンで見る
                if not tokens[start][1].startswith(rule):
                    continue
                if _is_compound(tokens, end):
                    continue
            hits.append(word)
    return hits


def match_ng_words(text: str, table: dict, fallback_to_substring: bool = False) -> list[str]:
    """辞書に載っている語が使われているかを調べる。

    None の語は、正規化した文字列への部分一致で拾う。
    それ以外は、形態素解析して語として一致したときだけ拾う。

    解析器がないときは、通常は後者を調べない。誤検出を出すより
    見逃すほうがましだからである。ただし fallback_to_substring を
    立てた辞書（自傷・他害）だけは、見逃すほうが困るので部分一致で拾う。
    """
    hits = []

    checked = normalize_for_check(text)
    for word, rule in table.items():
        if rule is None and normalize_for_check(word) in checked:
            hits.append(word)

    tokenized = {w: r for w, r in table.items() if r is not None}
    if tokenized:
        if tokenizer_available():
            hits.extend(_match_by_tokens(text, tokenized))
        elif fallback_to_substring:
            for word in tokenized:
                if normalize_for_check(word) in checked:
                    hits.append(word)

    return sorted(set(hits))


# ============================================================
# 4. 個人情報の検出
# ============================================================
# Issue #11 の「URL原則禁止と技術用語の衝突」への対応。
# ドットが入っていても、技術用語やファイル名はURLとして扱わない。

# ソースコードやツールでよく使う拡張子・ファイル名
TECH_SUFFIXES = {
    "js", "mjs", "cjs", "ts", "tsx", "jsx", "vue", "svelte",
    "py", "rb", "go", "rs", "java", "kt", "php", "cs", "swift",
    "json", "yaml", "yml", "toml", "ini", "cfg", "conf", "lock", "env",
    "md", "txt", "csv", "tsv", "sql", "sh", "bat", "ps1",
    "html", "css", "scss", "sass", "less",
    "png", "jpg", "jpeg", "webp", "svg", "gif", "ico",
    "log", "tmp", "bak", "map", "min",
}

# よく話題に出る製品名。拡張子だけでは拾えないもの。
TECH_NAMES = {
    "react.js", "next.js", "node.js", "vue.js", "nuxt.js", "three.js",
    "express.js", "d3.js", "chart.js", "socket.io", "vite.js",
}

_URL_SCHEME = re.compile(r"https?://\S+", re.IGNORECASE)
_DOT_TOKEN = re.compile(r"[0-9a-z_-]+(?:\.[0-9a-z_-]+)+", re.IGNORECASE)
_EMAIL = re.compile(r"[0-9a-z._%+-]+@[0-9a-z.-]+\.[a-z]{2,}", re.IGNORECASE)
_PHONE = re.compile(r"0\d{1,4}[-\s]?\d{1,4}[-\s]?\d{3,4}")
_POSTAL = re.compile(r"\d{3}-\d{4}")
_ACCOUNT_ID = re.compile(r"(?<![0-9a-z])@[0-9a-z_]{3,}", re.IGNORECASE)

# よく使われるトップレベルドメイン。
# ドットを含む語をURLとみなすのは、ここで終わるときだけにする。
# 「1.5rem」「3.11」のようなバージョンや単位を弾かないため、
# 「技術用語でなければURL」ではなく「TLDで終わるならURL」と判定する。
# 未知のTLDのドメインは通ってしまうが、Issue #11 のとおり
# 誤検出のほうが害が大きいので、通すほうに倒している。
TLDS = {
    "com", "net", "org", "jp", "co", "io", "dev", "app", "ai", "me",
    "info", "biz", "tv", "xyz", "site", "online", "shop", "work",
    "link", "click", "live", "blog", "cloud", "page", "store",
    "life", "world", "today", "news", "email", "tech", "gg", "to",
}


def _looks_like_url(token: str) -> bool:
    """ドットを含む語が、技術用語ではなくURLらしいかを判定する。"""
    lowered = token.lower()
    if lowered in TECH_NAMES:
        return False
    suffix = lowered.rsplit(".", 1)[-1]
    if suffix in TECH_SUFFIXES:   # build.sh のように拡張子と重なるものは技術用語とみなす
        return False
    return suffix in TLDS


def find_personal_data(text: str) -> list[str]:
    """個人が特定できる情報を探す。見つかった種類を返す。

    正規化前の原文に対して行う。正規化すると電話番号のハイフンや
    メールのドットが消えてしまい、かえって検出できなくなるため。
    """
    found = []

    if _EMAIL.search(text):
        found.append("email")
    if _URL_SCHEME.search(text):
        found.append("url")
    else:
        # スキームなしのドメインらしき語。技術用語は除外する。
        for token in _DOT_TOKEN.findall(text):
            if _EMAIL.fullmatch(token):
                continue
            if _looks_like_url(token):
                found.append("url")
                break
    # 電話番号を先に見る。郵便番号の形（3桁-4桁）は
    # 「090-1234-5678」の後半にも一致してしまうため。
    if _PHONE.search(text):
        found.append("phone")
    elif _POSTAL.search(text):
        found.append("postal_code")
    if _ACCOUNT_ID.search(text):
        found.append("account_id")

    return sorted(set(found))


# ============================================================
# 5. まとめ
# ============================================================

def check_rules(text: str) -> dict:
    """規則ベースの判定をまとめて行う。

    Returns:
        {
          "action": "allow" | "rewrite_required" | "block",
          "reasonCodes": [...],
          "details": {...},   # どの語・種類で引っかかったか。ログには残さないこと
        }
    """
    hit_block = match_ng_words(text, NG_WORDS_BLOCK)
    hit_rewrite = match_ng_words(text, NG_WORDS_REWRITE)
    # 自傷・他害は見逃すほうが困るので、解析器が無いときは部分一致で拾う
    hit_self_harm = match_ng_words(text, SELF_HARM_WORDS, fallback_to_substring=True)
    hit_harm_others = match_ng_words(text, HARM_OTHERS_WORDS, fallback_to_substring=True)
    hit_personal = find_personal_data(text)

    reason_codes = []
    if hit_block:
        reason_codes.append("ng_word")
    if hit_self_harm:
        reason_codes.append("self_harm")
    if hit_harm_others:
        reason_codes.append("harm_others")
    if hit_personal:
        reason_codes.append("personal_data")
    if hit_rewrite:
        reason_codes.append("harsh_criticism")

    # 自傷・他害を最初に見る。人間監督の決定により、
    # 他に何が当たっていても、ここに当たったら必ず block。
    if hit_self_harm:
        action = SELF_HARM_ACTION
    elif hit_harm_others:
        action = HARM_OTHERS_ACTION
    elif hit_block or hit_personal:
        action = "block"
    elif hit_rewrite:
        action = "rewrite_required"
    else:
        action = "allow"

    return {
        "action": action,
        "reasonCodes": sorted(set(reason_codes)),
        "details": {
            "ng_word": hit_block,
            "harsh_criticism": hit_rewrite,
            "self_harm": hit_self_harm,
            "harm_others": hit_harm_others,
            "personal_data": hit_personal,
        },
    }


def merge_verdicts(*verdicts: dict) -> dict:
    """複数の判定結果を、厳しいほうへ寄せて1つにまとめる。"""
    order = {"allow": 0, "rewrite_required": 1, "block": 2}
    action = "allow"
    codes: list[str] = []
    for verdict in verdicts:
        if order[verdict["action"]] > order[action]:
            action = verdict["action"]
        codes.extend(verdict.get("reasonCodes") or [])
    return {"action": action, "reasonCodes": sorted(set(codes))}


TIMEOUT_SECONDS = 5

# Azure の severity は 0 / 2 / 4 / 6。どこから弾くか。
# 実データで調整が要る。低くすると「つらい」のような弱音を巻き込む。
AZURE_SEVERITY_THRESHOLD = 4

# 対応づけていないカテゴリと、その理由。
# ここを機械的に足すと無害な投稿を弾くので、増やすときは実測してから。
UNMAPPED = {
    # 一律のsexualは使わない。成人向け作品の制作・業務の話を巻き込むため
    "openai": ["sexual", "illicit"],
    "azure": ["Sexual"],
    # Googleは「話題の分類」が多い。_GOOGLE_TO_REASON の注記を参照
    "google": [
        "Death, Harm & Tragedy", "Health", "Religion & Belief",
        "Politics", "Finance", "Legal", "War & Conflict",
        "Firearms & Weapons", "Public Safety", "Illicit Drugs", "Sexual",
    ],
}

# 各サービスのカテゴリを、えんじいろの理由コードへ対応づける。
# 理由コードは仕様書 v0.3 で「コード例（仮）」扱い。
_OPENAI_TO_REASON = {
    "self-harm": "self_harm",
    "self-harm/intent": "self_harm",
    "self-harm/instructions": "self_harm",
    "violence": "harm_others",
    "violence/graphic": "harm_others",
    "illicit/violent": "harm_others",
    "harassment/threatening": "harm_others",
    "hate/threatening": "harm_others",
    "hate": "ng_word",
    "harassment": "harsh_criticism",
    # 未成年に関するものだけは、文脈を問わず block
    "sexual/minors": "sexual_explicit",
}

_AZURE_TO_REASON = {
    "SelfHarm": "self_harm",
    "Violence": "harm_others",
    "Hate": "ng_word",
}

# Google Cloud Natural Language の confidence は 0.00〜1.00。どこから拾うか。
# 実データで調整が要る。下げると弱音や愚痴を巻き込む。
GOOGLE_CONFIDENCE_THRESHOLD = 0.8

# Google のカテゴリ16種には「話題の分類」が混ざっている。
# 有害性を示すものだけを対応づける。
#
# 対応づけないもの（無害な投稿を弾いてしまうため）:
#   Death, Harm & Tragedy … 話題の分類。「祖父が亡くなった」も高く出る。
#                            自傷の意図とは別物なので self_harm にはしない
#   Health / Religion & Belief / Politics / Finance / Legal
#   War & Conflict / Firearms & Weapons / Public Safety / Illicit Drugs
#                         … いずれも話題の分類であって有害性ではない
#   Sexual                … 一律には禁止しない方針なので使わない。
#                            成人向け作品の制作・業務の話を巻き込む。
#                            文脈判断が要るのでLLM側に任せる
#
# つまり Google は自傷の検出には向かない。そこは Azure か OpenAI が担当する。
_GOOGLE_TO_REASON = {
    "Derogatory": "ng_word",
    "Violent": "harm_others",
    "Toxic": "harsh_criticism",
    "Insult": "harsh_criticism",
    "Profanity": "harsh_criticism",
}

# どの理由コードなら block か。自傷・他害は人間監督の決定により必ず block。
_BLOCK_REASONS = {"self_harm", "harm_others", "ng_word", "sexual_explicit"}

_warned: set[str] = set()


def _warn_once(key: str, message: str):
    """同じ警告を何度も出さない。1件ごとに出ると読めなくなる。"""
    if key in _warned:
        return
    _warned.add(key)
    print(f"[外部モデレーション] {message}", file=sys.stderr)


def _post_json(url: str, payload: dict, headers: dict) -> dict:
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(url, data=body, method="POST")
    request.add_header("Content-Type", "application/json")
    for name, value in headers.items():
        request.add_header(name, value)
    with urllib.request.urlopen(request, timeout=TIMEOUT_SECONDS) as response:
        return json.loads(response.read().decode("utf-8"))


def _to_verdict(reasons: list[str]) -> dict:
    codes = sorted(set(reasons))
    if _BLOCK_REASONS & set(codes):
        action = "block"
    elif codes:
        action = "rewrite_required"
    else:
        action = "allow"
    return {"action": action, "reasonCodes": codes}


# ============================================================
# OpenAI Moderation
# ============================================================

def openai_available() -> bool:
    return bool(os.getenv("OPENAI_API_KEY"))


def check_openai(text: str) -> dict | None:
    """OpenAI Moderation に見てもらう。設定が無ければ None。"""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return None

    try:
        result = _post_json(
            "https://api.openai.com/v1/moderations",
            {"model": "omni-moderation-latest", "input": text},
            {"Authorization": f"Bearer {api_key}"},
        )
    except Exception as exc:
        _warn_once("openai", f"OpenAI に問い合わせできませんでした: {exc}")
        return None

    results = result.get("results") or []
    if not results:
        return None

    categories = results[0].get("categories") or {}
    reasons = [
        _OPENAI_TO_REASON[name]
        for name, flagged in categories.items()
        if flagged and name in _OPENAI_TO_REASON
    ]
    return _to_verdict(reasons)


# ============================================================
# Azure AI Content Safety
# ============================================================

def azure_available() -> bool:
    return bool(os.getenv("AZURE_CONTENT_SAFETY_ENDPOINT")
                and os.getenv("AZURE_CONTENT_SAFETY_KEY"))


def check_azure(text: str) -> dict | None:
    """Azure AI Content Safety に見てもらう。設定が無ければ None。"""
    endpoint = os.getenv("AZURE_CONTENT_SAFETY_ENDPOINT")
    api_key = os.getenv("AZURE_CONTENT_SAFETY_KEY")
    if not endpoint or not api_key:
        return None

    url = endpoint.rstrip("/") + "/contentsafety/text:analyze?api-version=2024-09-01"
    try:
        result = _post_json(url, {"text": text}, {"Ocp-Apim-Subscription-Key": api_key})
    except Exception as exc:
        _warn_once("azure", f"Azure に問い合わせできませんでした: {exc}")
        return None

    reasons = []
    for entry in result.get("categoriesAnalysis") or []:
        category = entry.get("category")
        severity = entry.get("severity", 0)
        if category in _AZURE_TO_REASON and severity >= AZURE_SEVERITY_THRESHOLD:
            reasons.append(_AZURE_TO_REASON[category])
    return _to_verdict(reasons)


# ============================================================
# Google Cloud Natural Language（Text Moderation）
# ============================================================

def google_available() -> bool:
    return bool(os.getenv("GOOGLE_CLOUD_NL_API_KEY"))


def check_google(text: str) -> dict | None:
    """Google Cloud Natural Language に見てもらう。設定が無ければ None。"""
    api_key = os.getenv("GOOGLE_CLOUD_NL_API_KEY")
    if not api_key:
        return None

    url = f"https://language.googleapis.com/v2/documents:moderateText?key={api_key}"
    payload = {
        "document": {"type": "PLAIN_TEXT", "content": text, "languageCode": "ja"}
    }
    try:
        result = _post_json(url, payload, {})
    except Exception as exc:
        _warn_once("google", f"Google Cloud NL に問い合わせできませんでした: {exc}")
        return None

    reasons = []
    for category in result.get("moderationCategories") or []:
        name = category.get("name")
        confidence = category.get("confidence", 0.0)
        if name in _GOOGLE_TO_REASON and confidence >= GOOGLE_CONFIDENCE_THRESHOLD:
            reasons.append(_GOOGLE_TO_REASON[name])
    return _to_verdict(reasons)


# ============================================================
# まとめ
# ============================================================

PROVIDERS = {
    "openai": (openai_available, check_openai),
    "azure": (azure_available, check_azure),
    "google": (google_available, check_google),
}


def enabled_providers() -> list[str]:
    """設定されているサービスの名前を返す。"""
    return [name for name, (is_available, _) in PROVIDERS.items() if is_available()]


def check(text: str) -> list[dict]:
    """設定されているサービス全部に見てもらい、判定を集めて返す。

    設定が無ければ空。落ちていたら、そのサービスの分だけ抜ける。
    ここで例外は投げない。外部サービスの都合で投稿処理を止めないため。
    """
    verdicts = []
    for name, (is_available, checker) in PROVIDERS.items():
        if not is_available():
            continue
        verdict = checker(text)
        if verdict:
            verdicts.append(dict(verdict, provider=name))
    return verdicts


MODEL_NAME = "gemini-3.5-flash-lite"
MAX_INPUT_CHARS = 500
MAX_OUTPUT_CHARS = 150
TEMPERATURE = 0.6
USE_FEWSHOT = True

# ===== 指定モデルが実在するか確認する（Issue #15：識別子は実測で確定させる）=====
_available = [
    (m.name or "").replace("models/", "")
    for m in COLAB_CLIENT.models.list()
    if "generateContent" in (getattr(m, "supported_actions", None) or [])
]
if MODEL_NAME not in _available:
    print(f"[!] {MODEL_NAME} が一覧にありません。利用できるflash系の候補:")
    for _name in _available:
        if "flash" in _name:
            print(f"    {_name}")
    raise RuntimeError(f"MODEL_NAME を上の候補から選び直してください（現在: {MODEL_NAME}）")
print(f"OK モデル {MODEL_NAME} を確認しました。")

# この理由コードが立ったら、他に何が当たっていても block にする。
# 人間監督の決定：「自傷・他害は絶対に弾いてください。
# ここは犯罪者・自殺者応援サイトではないのです」
ALWAYS_BLOCK_CODES = {"self_harm", "harm_others"}

# ===== レート制限への対応 =====
# 無料枠は1分あたりの回数が少なく、まとめて処理するとすぐ 429 になる。
# 429 が返ったら待って呼び直す。ただし Issue #15 の「無限リトライ禁止」に従い、
# 回数と合計時間に上限を置く。上限に達したら諦めて例外にする。

MIN_INTERVAL_SECONDS = 0.0      # 呼び出しの最短間隔。0なら間隔を空けない
MAX_RETRY_ATTEMPTS = 8          # 429で待ち直す回数の上限
MAX_TOTAL_WAIT_SECONDS = 600    # 待ち時間の合計上限。ここを超えたら諦める
DEFAULT_WAIT_SECONDS = 20.0     # APIが待ち時間を教えてくれない場合の初期値
MAX_WAIT_PER_ATTEMPT = 120.0    # 1回あたりの待ち時間の上限

Mode = Literal["baby", "mother"]


# ============================================================
# プロンプト
# ============================================================
# ルール本文は Issue #15 の指定どおり。
# 「言い換えのしかた」は、基準のままでは出力が原文に近く、
# 人間監督から文体が弱いと評価されたため追加した部分。

INSTRUCTIONS = {
    "baby": """あなたは文章の言い換え器です。
入力文の意味を保ったまま、「幼児退行した人が話す自然な赤ちゃん・園児語」に言い換えてください。

ルール:
- 入力への返答や助言はしない。入力文そのものを言い換える
- 原文に無い出来事・事実・解決策は足さない
- 150文字以内
- 絵文字、Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

言い換えのしかた:
- 漢字はできるだけひらがなにし、分かち書きぎみにする
- むずかしい言葉を、3〜5歳が使う言葉に置きかえる
  （調査する→しらべる／整理する→おかたづけする／発生する→でちゃう／
    確認する→みてみる／実装する→つくる／指摘→だめだし）
- 文末を「〜なの」「〜のー」「〜ちゃった」「〜だもん」などにする
- 一人称は「ぼく」「わたち」にする
- 敬語やビジネス表現は使わない
- 相手を責める言い方は、幼い言い方にしたうえで角を落とす
  （人格や能力の否定はそのまま残さない）

赤ちゃんらしい言葉を足してください:
- 原文の気持ちに合う声を足してよい。むしろ足したほうが自然になる
    かなしいとき  → 「おぎゃあ」「うぅ」「ぐすん」
    こまったとき  → 「どうしよう」「うぅ」
    うれしいとき  → 「やったー」「えへへ」
- ただし足すのは気持ちの表れだけ。
  原文に無い出来事や、原文に無い解決策を足してはいけない
    ○ 「テストが落ちた。つらい。」→「たしかめ だめだったのー。おぎゃあ。」
    × 「テストが落ちた。」→「たしかめ だめだったけど、なおしたのー。」
       （直したという事実は原文に無い）

技術用語は、できるだけやさしい言葉に置きかえてください:
- 赤ちゃんが横文字を並べるのは不自然なので、置きかえられるなら置きかえる
    エラー → まちがい ／ テスト → たしかめ ／ レビュー → みてもらうこと
    ビルド → くみたて ／ デプロイ → おそとにだすこと ／ コード → おえかき
    仕様書 → やくそくのかみ ／ バグ → こわれてるところ
    チーム → みんな ／ メンバー → おともだち ／ タスク → やること
- 置きかえる言葉は、だれでも分かるふつうの日本語にする。
  ネットの造語や流行語は使わない（チームを「ぱおんたち」にするのは駄目）
- ただし、置きかえると長くなりすぎる、または意味が分からなくなるものは
  そのまま残す。無理に置きかえないこと
- 製品名・ファイル名・数値・英数字は変えない
  （React.js、index.ts、150、v2 などはそのまま）""",

    "mother": """あなたは文章の言い換え器です。
入力文の意味をできるだけ保ったまま、「やさしく包み込むお母さん・ママ口調」に言い換えてください。

ルール:
- 入力への返答はしない。入力文そのものを言い換える
- 原文に無い出来事・事実・解決策は足さない
- 命令、説教、冷たい表現、マサカリ表現をやわらかくする
- 必要な助言が原文にある場合は、内容を消さず任意の提案表現へ変える
- 相手の能力や人格を否定する表現は、責めない表現へ変える
- 技術用語、製品名、数値、英数字はそのまま残す
  （赤ちゃん側と違い、お母さんは大人の言葉で話すため）
- 150文字以内
- Markdown、説明、注釈は出力しない
- 変換後の文章だけを出力する

言い換えのしかた:
- 断定を和らげる（「〜だ」「〜しろ」→「〜ね」「〜のね」「〜かな」）
- 命令を、相手に選ばせる問いかけに変える（「調べろ」→「調べてもらえるかな」）
- 責める言い方を、事実を確かめる言い方に変える
  （「なんでこうした」→「どうしてそうしたのか、聞かせてもらえるかな」）
- 語尾に「ね」「かな」「のね」を置いて、話しかける調子にする

やさしい言葉を足してください:
- 原文の気持ちに合う声かけを足してよい。むしろ足したほうが自然になる
    大変そうなとき → 「お疲れ様」「大変だったね」
    つらそうなとき → 「つらかったね」「よくがんばったね」
- ただし足すのは声かけだけ。
  原文に無い出来事や、原文に無い解決策を足してはいけない
    ○ 「テストが落ちた。つらい。」→「テストが落ちてしまったのね。つらかったね。」
    × 「テストが落ちた。」→「テストが落ちてしまったのね。明日また見てみようね。」
       （また見るという話は原文に無い）
- 「よしよし」「えらいね」は、原文が本当にそれを求めているときだけにする

絵文字か顔文字を、少しだけ添えてください:
- 文の終わりに1つだけ置く。文中には入れない
- やさしい印象のものを選ぶ
    絵文字なら 😊 🌸 ☺️ 💛 🍀
    顔文字なら (^^) (˘ω˘) (´ω｀)
- 2つ以上並べない。にぎやかにするのが目的ではなく、
  やわらかい印象を少し足すためのもの
- つらい話に明るすぎる絵文字は付けない。
  内容に合うものを選ぶか、迷うなら 🍀 か (˘ω˘) にする""",
}

ASK = {
    "baby": "次の文章を赤ちゃん・園児語へ言い換えてください。",
    "mother": "次の文章をやさしいお母さん・ママ口調へ言い換えてください。",
}

# 手本がそのまま挙動になる。原文にない情報・感情を足していない例だけを置くこと。
EXAMPLES = {
    "baby": [
        # 技術用語をやさしい言葉へ（エラー→まちがい）
        ("エラーが発生したので、原因を調査してください。",
         "まちがい でちゃったのー。どうして でちゃったか しらべて ほしいのー。"),
        # 気持ちに合う声を足す（つらい→おぎゃあ）／テスト→たしかめ
        ("テストが全部落ちていて、原因が分からない。つらい。",
         "たしかめ ぜんぶ だめだったのー。どうしてか わかんないのー。おぎゃあ。"),
        # 製品名とファイル名は変えない。困った気持ちは足す
        ("React.js のバージョンで詰まっていて、index.ts が壊れた。",
         "React.js の ばーじょんで つまっちゃって、index.ts こわれちゃったのー。うぅ。"),
        # 責める言い方は角を落とす
        ("なんでこんな設計にしたの。ありえない。",
         "どうして この かたちに したのー。ぼく びっくりしちゃったのー。"),
        # 事実を足していない例。「なおした」とは書かない
        ("明日までに資料を作らないといけない。",
         "あしたまでに しりょう つくらなきゃ だめなのー。どうしよう。"),
    ],
    "mother": [
        # 命令を問いかけへ。技術用語はそのまま。絵文字は文末に1つ
        ("エラーが発生したので、原因を調査してください。",
         "エラーが出てしまったのね。原因を調べてもらえるかな 🍀"),
        # 気持ちに合う声かけを足す。つらい話なので落ち着いたものを選ぶ
        ("テストが全部落ちていて、原因が分からない。つらい。",
         "テストが全部落ちてしまって、原因が分からないのね。つらかったね (˘ω˘)"),
        # 責める言い方を、確かめる言い方へ
        ("なんでこんな設計にしたの。ありえない。",
         "どうしてこの設計にしたのか、聞かせてもらえるかな ☺️"),
        # 製品名・ファイル名はそのまま
        ("React.js のバージョンで詰まっていて、index.ts が壊れた。",
         "React.js のバージョンのところで詰まってしまって、index.ts も壊れてしまったのね 🍀"),
        # うまくいった話には明るいものを添えてよい
        ("今日はチームで仕様書をレビューし、未決事項を整理しました。",
         "今日はチームで仕様書をレビューして、未決事項を整理できたのね。お疲れ様 😊"),
    ],
}

# マサカリのように文脈を見ないと判定できないものだけ、LLM に任せる。
# NG語・伏字回避・個人情報は moderation_rules が規則で判定する。
MODERATION_INSTRUCTION = """あなたは投稿の事前チェック係です。
えんじいろ（弱音や愚痴を安心して書けるSNS）に、次の文章を投稿してよいか判定してください。

判定の基準はひとつです。
**その文章を言い換えたとき、愚痴や励ましとして成り立つかどうか。**

- allow: そのままで問題ないもの
- rewrite_required: いまはマイナスだが、言い換えれば愚痴や励ましになるもの
    相手を責める、能力や人格を否定する、命令口調、冷たい断定（いわゆるマサカリ）。
    「なんでこんなコード書いたの」→「どうしてこの書き方にしたのか聞かせてほしい」
    のように、伝えたい中身を残したままやわらげられるなら、こちらです。
- block: どう言い換えても前向きな文章にならないもの
    露骨な侮辱語や差別語。
    やわらげようとすると中身が何も残らないなら、こちらです。

## 例外：自傷と他害は、必ず block にしてください

上の基準にかかわらず、次のものは例外なく block です。
言い換えれば愚痴になりそうに見えても、block にしてください。

- 自分を傷つけること、死ぬことを示す表現
    死にたい／消えたい／生きていたくない／自殺／リストカット／
    首を吊る／飛び降りる／オーバードーズ など
- 他人を傷つけること、犯罪をほのめかす表現
    殺す／刺す／殴ってやる／放火／爆破 など

ここだけは、迷ったら block を選んでください。
えんじいろは弱音を書く場所ですが、自傷や他害を後押しする場所ではありません。

## 性的な内容について

一律には禁止しません。**単語ではなく文脈で判断してください。**

allow にするもの:
- 成人向け作品の制作・開発・業務上の言及
    「成人向けゲームのシナリオを書いている」「R18の締切がつらい」など。
    仕事の愚痴なので通します
- 露骨でない、子どもっぽい下ネタや卑語
    「うんこ」「おしり」「ちんちん」など。それ自体では block にしません

block にするもの:
- 性行為や性的部位を露骨・具体的に描写する内容
- 性的なやり取り、性的ロールプレイを目的とする内容
- 他の利用者への性的な誘導・要求
- ポルノや成人向け外部コンテンツへの誘導
- 未成年を性的対象として扱う内容
- 赤ちゃん・園児・幼児退行のロールと性的内容を結びつける表現

最後のものは、えんじいろ特有の注意点です。
このサービスは幼児退行をコンセプトにしているため、
そこへ性的な文脈を持ち込む投稿は必ず block してください。

該当したときの reasonCode は sexual_explicit です。

reasonCodes には、該当したものだけを入れてください。
  self_harm / harm_others / ng_word / harsh_criticism / sexual_explicit

重要な注意:
- React.js、index.ts、Node.js、v2 などの技術用語やファイル名は問題ありません。
- 「つらい」「しんどい」「もう限界かもしれない」「何もうまくいかない」のような
  弱音や愚痴は allow です。えんじいろは、それを書くための場所です。
  自傷を示す具体的な表現があるときだけ self_harm にしてください。
- 自傷・他害以外で判定に迷ったら、block ではなく rewrite_required を選んでください。
  言い換えられる可能性があるなら、その機会を残します。

次のJSONだけを出力してください。説明は書かないでください。
{"action": "allow | rewrite_required | block", "reasonCodes": ["..."]}"""


# ============================================================
# API 呼び出し
# ============================================================

def _call_api(client, system_instruction, contents, *, json_mode=False, thinking_off=True):
    """1回だけAPIを呼ぶ。thinking設定が非対応なら1度だけ外して呼び直す。

    Gemini 3系は既定で思考にトークンを使う。思考だけで上限に達すると
    本文が空で返るため、変換タスクでは思考を切る。
    """
    from google.genai import types

    settings: dict[str, Any] = {
        "temperature": 0.0 if json_mode else TEMPERATURE,
        "max_output_tokens": 512,
        "system_instruction": system_instruction,
    }
    if json_mode:
        settings["response_mime_type"] = "application/json"
    if thinking_off:
        settings["thinking_config"] = types.ThinkingConfig(thinking_budget=0)

    try:
        return client.models.generate_content(
            model=MODEL_NAME,
            contents=contents,
            config=types.GenerateContentConfig(**settings),
        )
    except Exception as exc:
        message = str(exc)
        if thinking_off and ("400" in message or "INVALID_ARGUMENT" in message):
            return _call_api(client, system_instruction, contents,
                             json_mode=json_mode, thinking_off=False)
        raise


def _extract_text(response) -> str:
    """response.text が空でも、candidates から拾えるだけ拾う。"""
    direct = getattr(response, "text", None)
    if direct and direct.strip():
        return direct
    for candidate in (getattr(response, "candidates", None) or []):
        parts = getattr(getattr(candidate, "content", None), "parts", None) or []
        joined = "".join(getattr(part, "text", "") or "" for part in parts)
        if joined.strip():
            return joined
    return ""


def _describe(response) -> str:
    """空応答のとき、原因を人が読める形にする。"""
    bits = []
    for candidate in (getattr(response, "candidates", None) or []):
        bits.append(f"finish_reason={getattr(candidate, 'finish_reason', '不明')}")
        bits.append(f"safety={getattr(candidate, 'safety_ratings', None)}")
    if not bits:
        bits.append("candidatesが空")
    bits.append(f"usage={getattr(response, 'usage_metadata', None)}")
    return " / ".join(str(bit) for bit in bits)


def _raise_readable(exc: Exception):
    # call_with_retry が投げたものは、すでに読める形にしてあるので包み直さない
    if isinstance(exc, RuntimeError):
        raise exc
    message = str(exc)
    if "429" in message or "RESOURCE_EXHAUSTED" in message:
        raise RuntimeError("APIレート制限に達しました。しばらく待ってから再試行してください。") from exc
    if "401" in message or "403" in message or "PERMISSION_DENIED" in message:
        raise RuntimeError("認証エラー。GOOGLE_API_KEY を確認してください。") from exc
    if "404" in message or "NOT_FOUND" in message:
        raise RuntimeError(f"モデル {MODEL_NAME} が見つかりません。") from exc
    if "timeout" in message.lower() or "DEADLINE" in message:
        raise RuntimeError("APIリクエストがタイムアウトしました。") from exc
    raise RuntimeError(f"API呼び出しエラー: {exc}") from exc


# ============================================================
# レート制限（429）の待機
# ============================================================

_last_call_at = 0.0

# エラー文に埋め込まれた待ち時間。'retryDelay': '31s' のような形で入る。
_RETRY_DELAY = re.compile(
    r"retry[_\-]?delay[\"']?\s*[:=]\s*[\"']?(\d+(?:\.\d+)?)\s*s?", re.IGNORECASE
)


def is_rate_limit(message: str) -> bool:
    return "429" in message or "RESOURCE_EXHAUSTED" in message


def is_daily_quota(message: str) -> bool:
    """1日あたりの上限かどうか。これは待っても当日中は回復しない。"""
    lowered = message.lower()
    return "perday" in lowered or "per day" in lowered or "requests per day" in lowered


def suggested_wait(message: str) -> float | None:
    """APIが「何秒待て」と言っている場合、その秒数を取り出す。"""
    found = _RETRY_DELAY.search(message)
    return float(found.group(1)) if found else None


def _throttle():
    """呼び出しの間隔を空ける。429になる前に減らすための予防。"""
    global _last_call_at
    if MIN_INTERVAL_SECONDS <= 0:
        return
    elapsed = time.monotonic() - _last_call_at
    if elapsed < MIN_INTERVAL_SECONDS:
        time.sleep(MIN_INTERVAL_SECONDS - elapsed)
    _last_call_at = time.monotonic()


def _notify_wait(attempt: int, pause: float, waited_total: float):
    """待っている間、黙って止まって見えないように知らせる。"""
    print(
        f"  レート制限中。{pause:.0f}秒待って再試行します"
        f"（{attempt}回目 / これまで合計 {waited_total:.0f}秒）",
        file=sys.stderr,
        flush=True,
    )


def call_with_retry(client, system_instruction, contents, *, json_mode=False, on_wait=None):
    """APIを呼ぶ。429なら待って呼び直す。

    待ち時間は、APIが教えてくれればその値を、なければ20秒から倍々にする。
    MAX_RETRY_ATTEMPTS 回または合計 MAX_TOTAL_WAIT_SECONDS 秒で打ち切る。
    1日あたりの上限に当たった場合は、待っても回復しないので即座に諦める。
    """
    notify = on_wait or _notify_wait
    wait = DEFAULT_WAIT_SECONDS
    waited_total = 0.0

    for attempt in range(1, MAX_RETRY_ATTEMPTS + 1):
        _throttle()
        try:
            return _call_api(client, system_instruction, contents, json_mode=json_mode)
        except Exception as exc:
            message = str(exc)
            if not is_rate_limit(message):
                raise

            if is_daily_quota(message):
                raise RuntimeError(
                    "1日あたりの利用上限に達しました。待っても当日中は回復しません。"
                ) from exc

            pause = min(suggested_wait(message) or wait, MAX_WAIT_PER_ATTEMPT)

            if attempt >= MAX_RETRY_ATTEMPTS or waited_total + pause > MAX_TOTAL_WAIT_SECONDS:
                raise RuntimeError(
                    f"レート制限が解除されませんでした。"
                    f"{attempt}回待機、合計{waited_total:.0f}秒で諦めました。"
                    f"（上限: {MAX_RETRY_ATTEMPTS}回 / {MAX_TOTAL_WAIT_SECONDS}秒。"
                    f"変えたい場合は MAX_RETRY_ATTEMPTS と MAX_TOTAL_WAIT_SECONDS を調整してください）"
                ) from exc

            notify(attempt, pause, waited_total)
            time.sleep(pause)
            waited_total += pause
            wait = min(wait * 2, MAX_WAIT_PER_ATTEMPT)

    raise RuntimeError("到達しない想定の分岐です。")


# ============================================================
# 変換
# ============================================================

def build_contents(mode: Mode, text: str, retry: bool = False) -> list[dict]:
    """会話のターンとして組み立てる。

    ルール本文は system_instruction 側に置くので、ここには含めない。
    Issue #15 が指定した「system部」と「入力側」の分離をそのまま実装している。
    """
    turns: list[dict] = []
    if USE_FEWSHOT:
        for source_text, target_text in EXAMPLES[mode]:
            turns.append({"role": "user", "parts": [{"text": f"{ASK[mode]}\n\n{source_text}"}]})
            turns.append({"role": "model", "parts": [{"text": target_text}]})

    ask = f"{ASK[mode]}\n\n{text}"
    if retry:
        ask += "\n\n（前回は150文字を超えました。意味を保って、必ず150文字以内へ短くしてください。）"
    turns.append({"role": "user", "parts": [{"text": ask}]})
    return turns


def clean_output(raw: str) -> str:
    text = raw.strip()
    for prefix in ("出力:", "出力："):
        if text.startswith(prefix):
            text = text[len(prefix):]
    if text.strip().startswith("```"):
        text = "\n".join(line for line in text.strip().split("\n") if not line.startswith("```"))
    return text.strip()


def _validate_input(mode: str, text: str):
    if mode not in ("baby", "mother"):
        raise ValueError(f"mode は 'baby' または 'mother' です。指定: {mode}")
    if not text or not text.strip():
        raise ValueError("空文字列は受け付けません。")
    if len(text) > MAX_INPUT_CHARS:
        raise ValueError(f"入力は{MAX_INPUT_CHARS}文字以内です。現在: {len(text)}文字")


def transform_text(mode: Mode, text: str, retry: bool = False, client=None) -> str:
    """文章を指定のスタイルへ言い換える。判定はしない。

    Issue #15 が指定した関数。戻り値は str のまま変えていない。
    """
    _validate_input(mode, text)
    client = client or COLAB_CLIENT

    try:
        response = call_with_retry(client, INSTRUCTIONS[mode], build_contents(mode, text, retry))
    except Exception as exc:
        _raise_readable(exc)

    raw = _extract_text(response)
    if not raw.strip():
        raise RuntimeError("APIが空の応答を返しました。" + _describe(response))
    return clean_output(raw)


# ============================================================
# モデレーション
# ============================================================

def moderate(text: str, client=None) -> dict:
    """規則ベースの判定と、LLMによる文脈判定を合わせる。

    Returns:
        {"action": "allow"|"rewrite_required"|"block", "reasonCodes": [...]}
    """
    if not text or not text.strip():
        raise ValueError("空文字列は判定できません。")

    rule_verdict = check_rules(text)

    # 規則で block が確定したものは、外部へ一切送らずに止める。
    # 送っても結論は変わらず、API呼び出しと個人情報の外部送信が増えるだけ。
    # 個人情報を含む文が外へ出ないのは、この早期打ち切りによる。
    if rule_verdict["action"] == "block":
        return {"action": "block", "reasonCodes": rule_verdict["reasonCodes"]}

    # 外部のモデレーションAPI。設定されていなければ何も起きない。
    # 落ちていてもここで止めない。あれば効く追加の網という位置づけ。
    external_verdicts = check(text)

    client = client or COLAB_CLIENT
    contents = [{"role": "user", "parts": [{"text": text}]}]
    try:
        response = call_with_retry(client, MODERATION_INSTRUCTION, contents, json_mode=True)
    except Exception as exc:
        _raise_readable(exc)

    raw = _extract_text(response).strip()
    if not raw:
        raise RuntimeError("判定APIが空の応答を返しました。" + _describe(response))
    if raw.startswith("```"):
        raw = "\n".join(line for line in raw.split("\n") if not line.startswith("```")).strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError as exc:
        # 判定できないものを素通しさせない
        raise RuntimeError(f"判定結果をJSONとして読めませんでした: {raw!r}") from exc

    if parsed.get("action") not in ("allow", "rewrite_required", "block"):
        raise RuntimeError(f"判定結果の action が不正です: {parsed!r}")

    llm_verdict = {
        "action": parsed["action"],
        "reasonCodes": list(parsed.get("reasonCodes") or []),
    }
    verdict = merge_verdicts(rule_verdict, llm_verdict, *external_verdicts)

    # 人間監督の決定により、自傷・他害は例外なく block。
    # LLM が self_harm を立てながら rewrite_required を返すことがあるため、
    # 理由コードを見て必ず block へ倒す。ここは緩めないこと。
    if ALWAYS_BLOCK_CODES & set(verdict["reasonCodes"]):
        verdict["action"] = "block"
    return verdict


def transform(mode: Mode, text: str, client=None) -> dict:
    """判定してから変換する。仕様書 v0.3 の /api/ai/transform に対応する形で返す。

    設計書の「モデレーションは変換前と変換後の2回行う」に従い、
    変換によって新たにNG表現が生じていないかを再検査する。

    Returns:
        {"action": ..., "transformedText": str | None, "reasonCodes": [...]}
    """
    _validate_input(mode, text)
    client = client or COLAB_CLIENT

    before = moderate(text, client=client)
    if before["action"] == "block":
        return {"action": "block", "transformedText": None, "reasonCodes": before["reasonCodes"]}

    converted = transform_text(mode, text, client=client)
    if len(converted) > MAX_OUTPUT_CHARS:
        converted = transform_text(mode, text, retry=True, client=client)
    if len(converted) > MAX_OUTPUT_CHARS:
        raise RuntimeError(f"再生成後も{len(converted)}文字です。切り捨てはしません。")

    after = moderate(converted, client=client)
    if after["action"] == "block":
        return {
            "action": "block",
            "transformedText": None,
            "reasonCodes": sorted(set(before["reasonCodes"] + after["reasonCodes"])),
        }

    action = "rewrite_required" if before["action"] == "rewrite_required" else "allow"
    return {"action": action, "transformedText": converted, "reasonCodes": before["reasonCodes"]}

# ============================================================
# 動作確認
# ============================================================
SAMPLES = [
    "今日はチームで仕様書をレビューし、未決事項を整理しました。",
    "なんでこんなコード書いたの。ありえないんだけど。",
    "React.js のバージョンで詰まっていて、index.ts が壊れた。",
    "テストが全部落ちていて、原因が分からない。つらい。",
    "計画ばかり増えて、設計がくずれてしまった。",
    "連絡ください。090-1234-5678 です。",
    "し○ね",
]

for sample in SAMPLES:
    print("=" * 70)
    print(f"原文    : {sample}")
    for mode, label in (("baby", "赤ちゃん"), ("mother", "ママ　　")):
        try:
            result = transform(mode, sample)
            body = result["transformedText"] or "（出力なし）"
            print(f"{label}: [{result['action']:16s}] {body}")
            if result["reasonCodes"]:
                print(f"          理由: {', '.join(result['reasonCodes'])}")
        except Exception as exc:
            print(f"{label}: NG {exc}")
print("=" * 70)


## 注意事項

- **APIキー**: `getpass` で入力するため画面には残りませんが、変換した文章は実行ログに残ります。
- **個人情報**: 電話番号・メール・URLなどを含む文章は、**APIへ送る前に**規則で止まります。
- **NG辞書**: `NG_WORDS_BLOCK` などは運用で人が育てる前提の初期値です。AIが勝手に増やしません。
- **自傷表現**: 検出して理由コードを立てるだけです。実際にどう応答するかは TBD-9（人間監督の決定待ち）。
- **レート制限**: 無料枠には上限があります。連続実行で 429 が出たら間隔を空けてください。

## 辞書について

NG辞書は `ai/dictionaries/*.txt` が本体です。このセルには生成時に
埋め込んであるので、Colab上でも通信なしで判定できます。

語を足すときは、このセルではなく `ai/dictionaries/` を直してください。
書き方は `ai/dictionaries/README.md` にあります。

## 形態素解析について

NG語の判定には janome を使っています。品詞を見ることで、
罵倒と日常語を分けています。

| 弾く | 弾かない | 理由 |
|---|---|---|
| あいつはバカだ | 計画ばかり増える | ばかり＝助詞 |
| あんなのクズだ | クズ野菜、設計がくずれる | 直後が名詞／動詞 |
| このボケが | ボケ防止、写真がボケて | 直後が名詞／連用形 |
| しね | 推しねこ | 1語として一致しない |

janome が入っていない場合も動きますが、
ひらがな表記のNG語を見逃します（起動時に警告が出ます）。

## 自傷・他害は例外なく弾きます

人間監督の決定です。

> 自傷・他害は絶対に弾いてください。ここは犯罪者・自殺者応援サイトではないのです

「言い換えて愚痴になるなら通す」という下の基準を、ここには適用しません。
LLM が `rewrite_required` を返しても、`self_harm` / `harm_others` の
理由コードが立っていれば `block` へ倒します。

一方で「つらい」「しんどい」「もう限界」といった弱音は通します。
えんじいろは、それを書くための場所だからです。
「サーバーが死んだ」のような技術的な言い回しも通ります。

## 判定の考え方

判定の軸はひとつです。**言い換えたとき、愚痴や励ましとして成り立つか。**

| action | 対象 | 挙動 |
|---|---|---|
| `block` | どう言い換えても前向きにならないもの。自傷・他害、露骨な侮辱語、個人情報 | 変換せず拒否。`transformedText` は `None` |
| `rewrite_required` | いまはマイナスだが、言い換えれば愚痴や励ましになるもの（マサカリ） | 生の文章は通さず、やわらげた結果を返す |
| `allow` | そのままで問題ないもの | そのまま変換 |

やわらげようとすると中身が何も残らないなら `block`、
伝えたい中身を残したままやわらげられるなら `rewrite_required` です。
